# Customer Transactions Analysis — Exploratory Data Analysis

**Objective:** Clean, preprocess, and explore the `customers_data` and `transactions_data`
datasets to uncover patterns in customer profiles and transaction behavior, and to
surface any data quality issues that would affect further analysis.

**Datasets**
- `customers_data.csv` — one row per customer account
- `transactions_data.csv` — one row per transaction


In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display / plotting settings
sns.set_theme(style="whitegrid")

In [ ]:
#Load the data

# Customer data is TAB-separated
customers = pd.read_table('customers_data.csv')

# Transaction data is standard comma-separated
transactions = pd.read_csv("transactions_data.csv")

In [ ]:
customers.head()

In [ ]:
transactions.head()

In [ ]:
print(f"customers:    {customers.shape[0]:,} rows x {customers.shape[1]} columns")
print(f"transactions: {transactions.shape[0]:,} rows x {transactions.shape[1]} columns")

## Initial data overview

Check dtypes and null counts before deciding on a cleaning strategy.

In [ ]:
customers.info()


In [ ]:
# Null counts as a percentage
null_pct = (customers.isnull().mean() * 100).sort_values(ascending=False).round(2)
print(null_pct)

In [ ]:
transactions.info()


In [ ]:
null_pct_tx = (transactions.isnull().mean() * 100).sort_values(ascending=False).round(2)
print(null_pct_tx)


### Observations so far

- `card_status` (63%), `Restriciton Code` (29%) and `address` (14%) are heavily missing in `customers`.
  These likely mean "not applicable" (e.g. no card issued) rather than a data-entry error,
  so we will keep them and impute with an explicit "Unknown"/"None" label rather than dropping rows.
- `email_1` is missing for 68% of customers — too sparse to use for any analysis, will be dropped.
- `transactions` is essentially complete; only `narrative` (free text) and a handful of `Trans_ID`
  values are missing, and one `transactiondescription` is missing.


## Data cleaning - `customers`

In [ ]:
customers_clean = customers.copy()

# Fix column name typo for readability downstream
customers_clean = customers_clean.rename(columns={"Restriciton Code": "restriction_code"})

In [ ]:
# Parse dates
customers_clean["openingdate"] = pd.to_datetime(customers_clean["openingdate"], errors="coerce")
customers_clean["maxtransdate"] = pd.to_datetime(customers_clean["Maxtransdate"], errors="coerce")
customers_clean = customers_clean.drop(columns=["Maxtransdate"])

In [ ]:
# Drop columns
# email_1 is 68% empty region is constant across all 5,000 rows
print("Unique regions:", customers_clean["region"].unique())
customers_clean = customers_clean.drop(columns=["email_1", "region"])

In [ ]:
# Fill blanks 
for col in ["restriction_code", "card_status", "address", "product_group_desc"]:
    customers_clean[col] = customers_clean[col].fillna("Not Specified")

In [ ]:
# Duplicate check accountid
customers_clean.duplicated().sum()

In [ ]:
customers_clean.isnull().sum()

In [ ]:
# check the balance column for outliers
customers_clean["workingbalance"].describe()


`workingbalance` ranges from roughly -1.97 billion to +102 million

## Data cleaning - `transactions`

In [ ]:
transactions_clean = transactions.copy()

# Parse timestamp
transactions_clean["Trans_timestamp"] = pd.to_datetime(transactions_clean["Trans_timestamp"], format="mixed")

In [ ]:
# Standardize direction flag into a readable column, keep original too
transactions_clean["direction"] = transactions_clean["deb_cre_ind"].map({"D": "Debit", "C": "Credit"})

In [ ]:
# Fill missing narrative/description with explicit label rather than NaN ---
transactions_clean["narrative"] = transactions_clean["narrative"].fillna("No narrative provided")
transactions_clean["transactiondescription"] = transactions_clean["transactiondescription"].fillna("Unspecified")

In [ ]:
# Check AMOUNT_LCY column
transactions_clean["AMOUNT_LCY"].describe()

## Checking if the two tables can actually be joined

Before doing any merged analysis, verify how much overlap actually exists between the two files on the natural keys.

In [ ]:
common_accounts = set(customers_clean["accountid"]) & set(transactions_clean["accountid"])
common_customers = set(customers_clean["customer"]) & set(transactions_clean["customer"])

print(f"Accounts in customers file:     {customers_clean['accountid'].nunique():,}")
print(f"Accounts in transactions file:  {transactions_clean['accountid'].nunique():,}")
print(f"Accounts present in BOTH:       {len(common_accounts):,}")
print()
print(f"Customers in customers file:    {customers_clean['customer'].nunique():,}")
print(f"Customers in transactions file: {transactions_clean['customer'].nunique():,}")
print(f"Customers present in BOTH:      {len(common_customers):,}")


**Important data quality finding:** only ~868 accounts (roughly 17% of the customer file and
9% of the transactions file) appear in both datasets. This means the two files largely describe
*different* populations of accounts rather than being a clean one-to-many customer→transactions
relationship. Any merged/customer-level transaction analysis below is only valid for this small
overlapping subset — it is **not** representative of the full transaction volume, and this
limitation is called out again in the Insights section.

In [ ]:
# Build the overlap subset for any merged analysis - inner join on accountid
merged = transactions_clean.merge(customers_clean, on=["accountid", "customer"], how="inner")
print(len(merged))
merged[["accountid", "AMOUNT_LCY", "direction", "account_status", "customer_type", "sector"]].head()

## EDA - Customer profile

In [ ]:
customers_clean["customer_type"].value_counts(normalize=True).round(3)*100

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

customers_clean["account_status"].value_counts().plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Account Status Distribution")
axes[0].set_ylabel("Number of accounts")

customers_clean["customer_type"].value_counts().head(8).plot(kind="barh", ax=axes[1], color="#55A868")
axes[1].set_title("Top Customer Types")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


**65.2% of accounts are Dormant**, with only 29.1% Active
**Individual Customers** dominate the customer_type breakdown with 82.1%

In [ ]:
plt.figure(figsize=(10, 5))
customers_clean["product_group_desc"].value_counts().plot(kind="barh", color="#C44E52")
plt.title("Accounts by Product Group")
plt.xlabel("Number of accounts")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Digital channel adoption - what share of customers use each digital channel?
channel_adoption = customers_clean[["onebank", "onepay", "ussd"]].mean() * 100
channel_adoption.plot(kind="bar", color="#8172B2", figsize=(7, 5))
plt.title("Digital Channel Adoption Rate (%)")
plt.ylabel("% of accounts enrolled")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
channel_adoption.round(1)


Digital channel adoption is low across the board — each of `onebank`, `onepay` and `ussd`
sits well under half of the customer base, with `onepay` adoption the lowest. Combined with the
high dormancy rate above, this suggests digital enablement may be an area worth investigating
as a lever to reactivate dormant accounts.

In [ ]:
plt.figure(figsize=(9, 5))
# Log scale to handle the extreme balance outliers without them swallowing the chart
sns.histplot(customers_clean.loc[customers_clean["workingbalance"] > 0, "workingbalance"],
             log_scale=True, bins=40, color="#4C72B0")
plt.title("Working Balance Distribution (positive balances, log scale)")
plt.xlabel("Working balance (log scale)")
plt.tight_layout()
plt.show()

print("Accounts with balance <= 0:", (customers_clean["workingbalance"] <= 0).sum())


## EDA - Transaction behavior

In [ ]:
plt.figure(figsize=(6, 5))
transactions_clean["direction"].value_counts().plot(kind="bar", color=["#C44E52", "#55A868"])
plt.title("Debit vs Credit Transaction Count")
plt.ylabel("Number of transactions")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

transactions_clean["direction"].value_counts(normalize=True).round(3) * 100


In [ ]:
plt.figure(figsize=(10, 6))
transactions_clean["transactiondescription"].value_counts().head(10).plot(
    kind="barh", color="#4C72B0"
)
plt.title("Top 10 Transaction Types")
plt.xlabel("Count")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(transactions_clean["AMOUNT_LCY"], log_scale=True, bins=50, color="#8172B2")
plt.title("Transaction Amount Distribution (log scale)")
plt.xlabel("Amount (log scale)")
plt.tight_layout()
plt.show()


In [ ]:
# Daily transaction volume over time
daily_volume = transactions_clean.set_index("Trans_timestamp").resample("D").size()

plt.figure(figsize=(11, 5))
daily_volume.plot(color="#C44E52")
plt.title("Daily Transaction Volume")
plt.ylabel("Number of transactions")
plt.xlabel("Date")
plt.tight_layout()
plt.show()

print("Date range covered:", transactions_clean["Trans_timestamp"].min(), "to", transactions_clean["Trans_timestamp"].max())


The data covers roughly **one month** (late Aug to early Oct 2020), not a full year — this
should be stated explicitly in any report so readers don't assume year-round seasonality is
visible here.

## EDA - Merged view (overlap subset only)

In [ ]:
tx_per_account = merged.groupby("accountid").agg(n_transactions=("Trans_ID", "count"),total_amount=("AMOUNT_LCY", "sum"),).reset_index()

account_summary = customers_clean.merge(tx_per_account, on="accountid", how="inner")

plt.figure(figsize=(9, 5))
sns.boxplot(data=account_summary, x="account_status", y="n_transactions", color="#4C72B0")
plt.yscale("log")
plt.title("Transactions per Account by Account Status (overlap subset)")
plt.ylabel("Number of transactions (log scale)")
plt.tight_layout()
plt.show()

Active accounts clearly transact more — this confirms that activity
(not just status) is where revenue is generated. This supports prioritizing reactivation
campaigns specifically toward dormant accounts with partial engagement signals (e.g. accounts
that still have digital channels enrolled but low transaction count) as the easiest wins.

In [ ]:
account_summary.groupby("customer_type")["n_transactions"].agg(["count", "mean", "median"]).sort_values("count", ascending=False).head(8)

## Key Insights

1. **Dormancy is the biggest revenue opportunity.** ~65% of accounts are dormant vs. ~29%
   active. Converting even 10-15% of dormant accounts to active would meaningfully grow
   transaction revenue without new customer acquisition cost.
2. **Low digital adoption is likely a driver of dormancy.** Enrollment in onebank/onepay/ussd
   sits well under 50% across the board, digital convenience correlates with engagement, so
   this is a practical, controllable lever.
3. **Active accounts transact significantly more than dormant/inactive ones**, confirming that
   account status is a real behavioral signal the bank can target, not just a label.
4. **Debit transactions dominate (71%)**  withdrawals, transfers and POS purchases - suggesting
   customers primarily use accounts for spending/cash access rather than saving or investment
   products, which may point to cross-sell opportunities (savings plans, fixed deposits).
5. *(Data note)* Only ~868 of ~5,000 customer accounts could be matched to transaction records,
   so the behavioral findings above are directional(based on the matched subset) and should be
   validated against the full customer base once the data linkage gap is resolved.

## Recommendations

1. **Launch a targeted dormant-account reactivation campaign**, prioritizing customers who still
   have digital channels enrolled, they're the warmest leads, already one step engaged.
2. **Push digital channel enrollment** (onebank/onepay/ussd) as a low-cost engagement driver,
   especially bundled with reactivation outreach.
3. **Cross-sell savings/investment products** to active, debit-heavy customers as they're already
   engaged but currently using the account mainly for spending, not growth products.
4. **Resolve the data linkage gap** between customer and transaction systems so future targeting
   can be based on the full customer base, not just the ~17% currently matchable.

## 11. Conclusion

This EDA cleaned both datasets, quantified their missingness, and most importantly
surfaced a significant data linkage gap between the customer and transaction files that limits
how much can be concluded from a merged view. Within those limits, the customer base skews
heavily dormant with low digital adoption, balances are highly skewed with some clearly
implausible extreme values, and transaction activity (dominated by transfers, withdrawals and
POS purchases) is concentrated in a minority of accounts over a one-month window.